# Calibración

## Checkerboard usado

| Parámetro | Valor |
|---|---|
| Esquinas internas (columnas × filas) | 9 × 6 (54 puntos) |
| Cantidad de cuadrados (columnas × filas) | 10 × 7 |
| Lado de cada cuadrado | 28 mm (2,8 cm) |
| Tamaño del patrón, sin márgenes | 280 × 196 mm |

In [60]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))

## Imports de módulos

In [61]:
import pickle
import cv2
import numpy as np

from src.utils.prints import print_matrix

## Carga de datos

Contenidos de stereo_calibration.pkl

| Clave | Tipo / forma | Qué contiene |
|---|---|---|
| `left_K` | array `(3, 3)` | Matriz de parámetros intrínsecos de la cámara izquierda |
| `right_K` | array `(3, 3)` | Matriz de parámetros intrínsecos de la cámara derecha |
| `left_dist` | array `(1, 5)` | Coeficientes de distorsión de la izquierda |
| `right_dist` | array `(1, 5)` | Coeficientes de distorsión de la derecha |
| `R` | array `(3, 3)` | Rotación entre las cámaras |
| `T` | array `(3, 1)` | Traslación entre las cámaras |
| `E` | array `(3, 3)` | Matriz esencial |
| `F` | array `(3, 3)` | Matriz fundamental |
| `image_size` | tupla `(1920, 1080)` | Tamaño de imagen: `(ancho, alto)` |

---

Contenidos de stereo_maps.pkl

| Clave | Tipo / forma | Qué contiene |
|---|---|---|
| `left_map_x` | array `(1080, 1920)`, `float32` | Coordenadas X de origen para rectificar la imagen izquierda |
| `left_map_y` | array `(1080, 1920)`, `float32` | Coordenadas Y de origen para rectificar la imagen izquierda |
| `right_map_x` | array `(1080, 1920)`, `float32` | Coordenadas X de origen para rectificar la imagen derecha |
| `right_map_y` | array `(1080, 1920)`, `float32` | Coordenadas Y de origen para rectificar la imagen derecha |
| `R1` | array `(3, 3)`, `float64` | Rotación de rectificación izquierda |
| `R2` | array `(3, 3)`, `float64` | Rotación de rectificación derecha |
| `P1` | array `(3, 4)`, `float64` | Matriz de proyección izquierda rectificada |
| `P2` | array `(3, 4)`, `float64` | Matriz de proyección derecha rectificada |
| `Q` | array `(4, 4)`, `float64` | Matriz para convertir disparidad en coordenadas 3D |
| `validRoi1` | tupla `(0, 1, 1920, 1079)` | Región válida izquierda: `(x, y, ancho, alto)` |
| `validRoi2` | tupla `(0, 0, 1920, 1080)` | Región válida derecha: `(x, y, ancho, alto)` |

In [62]:
CALIBRATION_DIR = ROOT_DIR / "data/calibration_results"

with (CALIBRATION_DIR / "stereo_calibration.pkl").open("rb") as f:
    calib: dict = pickle.load(f)

with (CALIBRATION_DIR / "stereo_maps.pkl").open("rb") as f:
    maps: dict = pickle.load(f)

## 1. Parámetros de calibración

In [ ]:
for k, v in calib.items():
    if k == "image_size":
        print(f"{k} = {v}")
        continue
    print_matrix(name=k, matrix=v)

print(f"baseline = {np.linalg.norm(calib.get("T"))}")

587.482,0,990.671
0,593.957,552.905
0,0,1


-0.026457,0.00706564,0.00214563,0.00358874,-0.00371444


602.448,0,931.245
0,597.977,516.344
0,0,1


0.0114667,-0.0386206,-0.000216791,-0.0127469,0.00601503


0.997875,0.00114852,0.0651462
-0.00357237,0.999305,0.037102
-0.0650583,-0.0372559,0.997186


-56.8894
1.85426
9.35663


-0.08721,-9.41921,1.5019
5.63561,-2.10872,57.3389
-1.64709,-56.852,-2.23151


4.37342e-08,4.67206e-06,-0.00306901
-2.84729e-06,1.05378e-06,-0.0147809
0.00192707,0.0120937,1


image_size = (1920, 1080)
baseline = 57.683564136797926


## 2. Error de reproyección